# 🔥 PINN — Inverse Heat Source (1D) — v5
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/pinn-inverse-heat/blob/main/notebooks/01_inverse_1D_colab.ipynb)

$$-\frac{d^2T}{dx^2} = f(x), \quad x\in(0,1), \quad T(0)=T(1)=0$$

## Root cause of previous failures

The Tikhonov weight `w_reg` was **way too large**: the regularisation loss dominated the fit loss by a factor ~10 000, forcing $\mathcal{N}_f$ to stay flat (smooth ≈ constant).  
Rule of thumb: `w_reg * ||df/dx||²` should be **at most 10% of the fit loss**, not 10 000× larger.

| Version | w_reg Phase 2 | Result |
|---------|--------------|--------|
| v3/v4 | 1e-3 | Tikhonov >> Fit → f flat |
| **v5** | **1e-7** | Balanced → f converges |

Phase 3 also gets a cleaner formulation without Tikhonov (already handled in Phase 2).

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 1 — Colab setup
# ══════════════════════════════════════════════════
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repo to get src/ modules
    if not os.path.exists('/content/pinn-inverse-heat'):
        !git clone https://github.com/MaximeAuger/pinn-inverse-heat.git
    sys.path.insert(0, '/content/pinn-inverse-heat/src')
    
    # Create results dir in Colab's writable space
    RESULTS_DIR = '/content/results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running on Colab. Results → {RESULTS_DIR}')
else:
    sys.path.insert(0, '../src')
    RESULTS_DIR = '../results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running locally. Results → {RESULTS_DIR}')

In [ ]:
# ══════════════════════════════════════════════
#  CELL 2 — Imports & helpers
# ══════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

torch.manual_seed(42)
np.random.seed(42)

def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def set_grad(model, flag):
    for p in model.parameters(): p.requires_grad_(flag)

print(f'PyTorch {torch.__version__}')

In [ ]:
# ══════════════════════════════════════════════
#  CELL 3 — Architecture
# ══════════════════════════════════════════════
class MLP(nn.Module):
    def __init__(self, layers, init_scale=1.0):
        super().__init__()
        net = []
        for i in range(len(layers)-1):
            lin = nn.Linear(layers[i], layers[i+1])
            nn.init.xavier_normal_(lin.weight)
            lin.weight.data *= init_scale
            nn.init.zeros_(lin.bias)
            net.append(lin)
            if i < len(layers)-2:
                net.append(nn.Tanh())
        self.net = nn.Sequential(*net)

    def forward(self, x): return self.net(x)

torch.manual_seed(42)
pinn       = MLP([1, 64, 64, 64, 1], init_scale=1.0)
source_net = MLP([1, 64, 64, 64, 1], init_scale=0.1)
print('Models created ✓')

In [ ]:
# ══════════════════════════════════════════════
#  CELL 4 — Data
# ══════════════════════════════════════════════
def f_true(x): return np.sin(np.pi*x) + 0.5*np.sin(3*np.pi*x)
def T_true(x): return (1/np.pi**2)*np.sin(np.pi*x) + (0.5/(9*np.pi**2))*np.sin(3*np.pi*x)

N_OBS = 20; NOISE = 0.01
x_obs_np = np.linspace(0.05, 0.95, N_OBS)
T_clean  = T_true(x_obs_np)
T_obs_np = T_clean + NOISE*np.max(np.abs(T_clean))*np.random.randn(N_OBS)

x_obs  = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1)
T_obs  = torch.tensor(T_obs_np, dtype=torch.float32).unsqueeze(1)
x_col  = torch.linspace(0, 1, 200).unsqueeze(1)
x_eval = torch.linspace(0, 1, 400).unsqueeze(1)
x_plot = np.linspace(0, 1, 400)
x0     = torch.zeros(1,1); x1 = torch.ones(1,1)

print(f'{N_OBS} observations, noise={NOISE*100:.0f}% ✓')

## Phase 1 — Fit $\mathcal{N}_T$ from data
Pure data regression + BCs. No PDE, no source network involved.

In [ ]:
# ══════════════════════════════════════════════
#  CELL 5 — Phase 1
# ══════════════════════════════════════════════
set_grad(pinn, True); set_grad(source_net, False)

opt1  = optim.Adam(pinn.parameters(), lr=1e-3)
sch1  = optim.lr_scheduler.ReduceLROnPlateau(opt1, patience=500, factor=0.5, min_lr=1e-6, verbose=False)
hist1 = []
PHASE1_EPOCHS = 10_000

print('Phase 1 — fitting T(x) from data')
for ep in range(1, PHASE1_EPOCHS+1):
    opt1.zero_grad()
    L_bc   = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2
    L_data = torch.mean((pinn(x_obs) - T_obs)**2)
    loss   = 500.0*L_bc + 1000.0*L_data
    loss.backward()
    opt1.step(); sch1.step(loss)
    hist1.append(loss.item())
    if ep % 2000 == 0:
        print(f'  ep {ep:6d} | loss {loss.item():.3e} | lr {opt1.param_groups[0]["lr"]:.1e}')
        free()

with torch.no_grad():
    T_phase1 = pinn(x_eval).squeeze().numpy().copy()
T_l2_1 = np.mean((T_phase1 - T_true(x_plot))**2)**0.5
print(f'Phase 1 done ✓  T L² = {T_l2_1:.2e}')

plt.figure(figsize=(7,4))
plt.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='$T^*$')
plt.plot(x_plot, T_phase1, 'r--', lw=2, label='$\\mathcal{N}_T$ Phase 1')
plt.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5)
plt.title(f'Phase 1  |  T L² = {T_l2_1:.2e}'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase1_T.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free()

## Phase 2 — Identify $f$ from frozen $\mathcal{N}_T$

We fit $\mathcal{N}_f$ to $-\hat{T}''$.

### ⚠️ Key fix: Tikhonov weight

The Tikhonov term regularises the **gradient** of $f$. Its weight must satisfy:
$$w_{reg} \cdot \|f'\|^2 \ll L_{fit}$$

We monitor both terms at each step and print their ratio. A healthy ratio is < 0.1.

In [ ]:
# ══════════════════════════════════════════════
#  CELL 6 — Phase 2  (fixed Tikhonov weight)
# ══════════════════════════════════════════════
set_grad(pinn, False)        # FROZEN
set_grad(source_net, True)

# ── Correct Tikhonov weight ─────────────────────────────────
# Rule: w_reg * ||df/dx||^2 should be << L_fit
# Empirically: ||df/dx||^2 ~ O(pi^2) for sin(pi*x) source
# L_fit starts at ~ 1 (random init) → w_reg = 1e-7 gives ratio ~1e-6 initially
W_REG_P2 = 1e-7

opt2  = optim.Adam(source_net.parameters(), lr=1e-3)
sch2  = optim.lr_scheduler.ReduceLROnPlateau(opt2, patience=500, factor=0.5, min_lr=1e-7, verbose=False)
hist2 = []
PHASE2_EPOCHS = 15_000
snapshots_p2  = []

print('Phase 2 — identifying f(x) = -T\'\'(x)')
print(f'  w_reg = {W_REG_P2:.0e}  (was 1e-3 in v4 — 10^6 times smaller)')
print()

for ep in range(1, PHASE2_EPOCHS+1):
    opt2.zero_grad()

    # Compute -T'' with frozen N_T
    x_c  = x_col.detach().clone().requires_grad_(True)
    T_c  = pinn(x_c)
    dT   = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
    d2T  = torch.autograd.grad(dT,  x_c, torch.ones_like(dT),  create_graph=False)[0]
    tgt  = -d2T.detach()          # target: f ≈ -T''

    # Fit N_f to target
    f_p  = source_net(x_col)
    L_fit = torch.mean((f_p - tgt)**2)

    # Tikhonov — penalise ||df/dx||² with small weight
    x_r  = x_col.detach().clone().requires_grad_(True)
    f_r  = source_net(x_r)
    df   = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
    L_reg = torch.mean(df**2)

    loss = L_fit + W_REG_P2 * L_reg
    loss.backward()
    opt2.step(); sch2.step(loss)

    hist2.append({'fit': L_fit.item(), 'reg': L_reg.item(),
                  'reg_weighted': (W_REG_P2*L_reg).item()})

    if ep % 1000 == 0 or ep == 1:
        with torch.no_grad():
            f_s = source_net(x_eval).squeeze().numpy().copy()
        snapshots_p2.append((ep, x_eval.squeeze().numpy().copy(), f_s))

    if ep % 3000 == 0:
        ratio = (W_REG_P2*L_reg.item()) / (L_fit.item() + 1e-12)
        print(f'  ep {ep:6d} | L_fit {L_fit.item():.3e} | '
              f'w*L_reg {W_REG_P2*L_reg.item():.3e} | ratio {ratio:.3f} | '
              f'lr {opt2.param_groups[0]["lr"]:.1e}')
        free()

with torch.no_grad():
    f_phase2 = source_net(x_eval).squeeze().numpy().copy()
f_l2_2 = np.mean((f_phase2 - f_true(x_plot))**2)**0.5
print(f'\nPhase 2 done ✓  f L² = {f_l2_2:.2e}')

plt.figure(figsize=(7,4))
plt.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='$f^*$ true')
plt.plot(x_plot, f_phase2, 'r--', lw=2, label='$\\mathcal{N}_f$ Phase 2')
plt.title(f'Phase 2  |  f L² = {f_l2_2:.2e}'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase2_f.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free()

In [ ]:
# ══════════════════════════════════════════════
#  CELL 7 — Phase 2 diagnostics
#  Check that Tikhonov is NOT dominating
# ══════════════════════════════════════════════
ep_ax = np.arange(1, PHASE2_EPOCHS+1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogy(ep_ax, [h['fit'] for h in hist2], 'b-', lw=1.2, label='L_fit')
axes[0].semilogy(ep_ax, [h['reg_weighted'] for h in hist2], 'r--', lw=1.2, label=f'w_reg·L_reg  (w={W_REG_P2:.0e})')
axes[0].set_title('Phase 2 — loss terms (must NOT cross)'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

ratio = [h['reg_weighted']/(h['fit']+1e-12) for h in hist2]
axes[1].semilogy(ep_ax, ratio, 'g-', lw=1.5)
axes[1].axhline(0.1, color='red', ls='--', label='ratio=0.1 threshold')
axes[1].set_title('Tikhonov/Fit ratio  (healthy < 0.1)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Phase 2 Diagnostics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase2_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free()

## Phase 3 — Joint fine-tuning

**Key fix**: No Tikhonov in Phase 3 — both networks are already well-initialised and we just need to couple them through the PDE.  
The PDE loss *is* the coupling: it forces $\mathcal{N}_f = -\mathcal{N}_T''$, which is the smoothest consistent solution.

In [ ]:
# ══════════════════════════════════════════════
#  CELL 8 — Phase 3  (no Tikhonov)
# ══════════════════════════════════════════════
set_grad(pinn, True); set_grad(source_net, True)

all_params = list(pinn.parameters()) + list(source_net.parameters())
opt3  = optim.Adam(all_params, lr=5e-5)
sch3  = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=10_000, eta_min=1e-7)
hist3 = []
snapshots_p3 = []
PHASE3_EPOCHS = 10_000

# Balanced weights — no Tikhonov
W_PDE  = 10.0
W_BC   = 500.0
W_DATA = 100.0

print('Phase 3 — joint fine-tuning  (no Tikhonov)')
print(f'  w_pde={W_PDE}  w_bc={W_BC}  w_data={W_DATA}  w_reg=0')
print()

for ep in range(1, PHASE3_EPOCHS+1):
    opt3.zero_grad()

    # PDE coupling
    x_c  = x_col.detach().clone().requires_grad_(True)
    T_c  = pinn(x_c)
    dT   = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
    d2T  = torch.autograd.grad(dT,  x_c, torch.ones_like(dT),  create_graph=True)[0]
    f_c  = source_net(x_col)
    L_pde = torch.mean((d2T + f_c)**2)

    # BCs
    L_bc   = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2

    # Data
    L_data = torch.mean((pinn(x_obs) - T_obs)**2)

    loss = W_PDE*L_pde + W_BC*L_bc + W_DATA*L_data
    loss.backward()
    torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
    opt3.step(); sch3.step()

    hist3.append({'total': loss.item(), 'pde': L_pde.item(),
                  'bc': L_bc.item(), 'data': L_data.item()})

    if ep % 500 == 0 or ep == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).squeeze().numpy().copy()
            f_s = source_net(x_eval).squeeze().numpy().copy()
        snapshots_p3.append((ep, x_eval.squeeze().numpy().copy(), T_s, f_s))

    if ep % 2000 == 0:
        print(f'  ep {ep:6d} | pde {L_pde.item():.3e} | data {L_data.item():.3e} | bc {L_bc.item():.3e}')
        free()

with torch.no_grad():
    T_final = pinn(x_eval).squeeze().numpy().copy()
    f_final = source_net(x_eval).squeeze().numpy().copy()
snapshots_p3.append(('final', x_eval.squeeze().numpy().copy(), T_final, f_final))

T_err = np.abs(T_final - T_true(x_plot))
f_err = np.abs(f_final - f_true(x_plot))
print(f'\nPhase 3 done ✓')
print(f'  T L² = {np.mean(T_err**2)**0.5:.2e}')
print(f'  f L² = {np.mean(f_err**2)**0.5:.2e}')
free()

In [ ]:
# ══════════════════════════════════════════════
#  CELL 9 — Final results
# ══════════════════════════════════════════════
x_np = x_eval.squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='Analytical $T^*$')
ax.plot(x_np, T_final, 'r--', lw=2, label='PINN')
ax.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, alpha=0.8, label='Obs.')
ax.set_title('Temperature $T(x)$', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
T_l2 = np.mean(T_err**2)**0.5
ax.text(0.05, 0.95, f'Max: {T_err.max():.2e}\nL²: {T_l2:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontsize=10)

ax = axes[1]
ax.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='True $f^*$')
ax.plot(x_np, f_final, 'r--', lw=2, label='PINN recovered')
ax.set_title('Source $f(x)$ — recovered', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
f_l2 = np.mean(f_err**2)**0.5
ax.text(0.05, 0.95, f'Max: {f_err.max():.2e}\nL²: {f_l2:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontsize=10)

plt.suptitle('PINN Inverse Problem — Final Results (v5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/final_results.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free()

In [ ]:
# ══════════════════════════════════════════════
#  CELL 10 — Loss history (3 phases)
# ══════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].semilogy(hist1, 'b-', lw=1.2)
axes[0].set_title('Phase 1 — T regression'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].semilogy([h['fit'] for h in hist2], 'b-', lw=1.2, label='L_fit')
axes[1].semilogy([h['reg_weighted'] for h in hist2], 'r--', lw=1.2, label=f'w·L_reg')
axes[1].set_title('Phase 2 — f identification'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

for k, ls, c in [('pde','-','b'),('bc','--','orange'),('data','-.','g')]:
    axes[2].semilogy([h[k] for h in hist3], ls=ls, color=c, lw=1.2, label=k.upper())
axes[2].set_title('Phase 3 — joint fine-tuning'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Loss History — 3 Phases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_history.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free()

In [ ]:
# ══════════════════════════════════════════════
#  CELL 11 — Training animation (Phase 3)
# ══════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

def animate(i):
    lbl, xs, Ts, fs = snapshots_p3[i]
    for ax in axes: ax.cla()
    axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2, alpha=0.5, label='$T^*$')
    axes[0].plot(xs, Ts, 'r-', lw=2, label='PINN')
    axes[0].scatter(x_obs_np, T_obs_np, c='gray', s=25, zorder=5)
    axes[0].set_ylim(-0.005, 0.14)
    axes[0].set_title(f'T(x) — ep {lbl}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(x_plot, f_true(x_plot), 'b-', lw=2, alpha=0.5, label='$f^*$')
    axes[1].plot(xs, fs, 'r-', lw=2, label='PINN')
    axes[1].set_ylim(-0.5, 1.5)
    axes[1].set_title(f'f(x) — ep {lbl}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()

ani = animation.FuncAnimation(fig, animate, frames=len(snapshots_p3), interval=200)
gif_path = f'{RESULTS_DIR}/training_animation.gif'
ani.save(gif_path, writer='pillow', fps=4, dpi=80)
plt.close(); free()
print(f'GIF saved ({os.path.getsize(gif_path)//1024} KB)')
from IPython.display import Image; Image(gif_path)

In [ ]:
# ══════════════════════════════════════════════
#  CELL 12 — Noise sensitivity
# ══════════════════════════════════════════════
def sequential_train(x_obs_t, T_obs_t, ep1=8000, ep2=12000, ep3=8000):
    torch.manual_seed(0)
    _p = MLP([1,64,64,64,1], init_scale=1.0)
    _s = MLP([1,64,64,64,1], init_scale=0.1)

    # Phase 1
    set_grad(_p,True); set_grad(_s,False)
    _o = optim.Adam(_p.parameters(), lr=1e-3)
    for ep in range(ep1):
        _o.zero_grad()
        loss = 500*(_p(x0).squeeze()**2+_p(x1).squeeze()**2) + 1000*torch.mean((_p(x_obs_t)-T_obs_t)**2)
        loss.backward(); _o.step()
        if ep%2000==0: free()

    # Phase 2
    set_grad(_p,False); set_grad(_s,True)
    _o2 = optim.Adam(_s.parameters(), lr=1e-3)
    for ep in range(ep2):
        _o2.zero_grad()
        x_c=x_col.detach().clone().requires_grad_(True)
        T_c=_p(x_c)
        dT=torch.autograd.grad(T_c,x_c,torch.ones_like(T_c),create_graph=True)[0]
        d2T=torch.autograd.grad(dT,x_c,torch.ones_like(dT),create_graph=False)[0]
        f_p=_s(x_col)
        x_r=x_col.detach().clone().requires_grad_(True)
        f_r=_s(x_r)
        df=torch.autograd.grad(f_r,x_r,torch.ones_like(f_r),create_graph=True)[0]
        loss=torch.mean((f_p+d2T.detach())**2) + 1e-7*torch.mean(df**2)
        loss.backward(); _o2.step()
        if ep%2000==0: free()

    # Phase 3
    set_grad(_p,True); set_grad(_s,True)
    _params=list(_p.parameters())+list(_s.parameters())
    _o3=optim.Adam(_params,lr=5e-5)
    for ep in range(ep3):
        _o3.zero_grad()
        x_c=x_col.detach().clone().requires_grad_(True)
        T_c=_p(x_c)
        dT=torch.autograd.grad(T_c,x_c,torch.ones_like(T_c),create_graph=True)[0]
        d2T=torch.autograd.grad(dT,x_c,torch.ones_like(dT),create_graph=True)[0]
        f_c=_s(x_col)
        loss=10*torch.mean((d2T+f_c)**2)+500*(_p(x0).squeeze()**2+_p(x1).squeeze()**2)+100*torch.mean((_p(x_obs_t)-T_obs_t)**2)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(_params,1.0)
        _o3.step()
        if ep%2000==0: free()

    with torch.no_grad(): f_pred=_s(x_eval).squeeze().numpy().copy()
    del _p,_s,_params; free()
    return f_pred

noise_levels = [0.0, 0.01, 0.02, 0.05, 0.10]
results_noise = {}
x_np = x_eval.squeeze().numpy()

for noise in noise_levels:
    np.random.seed(0)
    T_n = T_clean + noise*np.max(np.abs(T_clean))*np.random.randn(N_OBS)
    f_pred = sequential_train(x_obs, torch.tensor(T_n,dtype=torch.float32).unsqueeze(1))
    l2 = float(np.mean((f_pred-f_true(x_np))**2)**0.5)
    results_noise[noise] = (f_pred, l2)
    print(f'  noise={noise*100:5.1f}%  →  f L² = {l2:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.plasma
axes[0].plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$', zorder=10)
for i,(n,(fp,_)) in enumerate(results_noise.items()):
    axes[0].plot(x_np, fp, '--', lw=1.5, color=cmap(i/len(noise_levels)), label=f'{n*100:.0f}%')
axes[0].set_title('Source recovery vs noise'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([n*100 for n in noise_levels],[results_noise[n][1] for n in noise_levels],'ro-',ms=8)
axes[1].set_xlabel('Noise (%)'); axes[1].set_ylabel('L² error'); axes[1].grid(alpha=0.3)
axes[1].set_title('L² error vs noise')
plt.suptitle('Noise Sensitivity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/noise_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free()

In [ ]:
# ══════════════════════════════════════════════
#  CELL 13 — Tikhonov study
# ══════════════════════════════════════════════
# Note: now we study w_reg in the CORRECT range
reg_weights = [0.0, 1e-8, 1e-7, 1e-5]
results_reg = {}
np.random.seed(0)
T_noisy   = T_clean + 0.03*np.max(np.abs(T_clean))*np.random.randn(N_OBS)
T_noisy_t = torch.tensor(T_noisy, dtype=torch.float32).unsqueeze(1)

for wr in reg_weights:
    torch.manual_seed(0)
    _p = MLP([1,64,64,64,1], init_scale=1.0)
    _s = MLP([1,64,64,64,1], init_scale=0.1)
    set_grad(_p,True); set_grad(_s,False)
    _o=optim.Adam(_p.parameters(), lr=1e-3)
    for ep in range(8000):
        _o.zero_grad()
        loss=500*(_p(x0).squeeze()**2+_p(x1).squeeze()**2)+1000*torch.mean((_p(x_obs)-T_noisy_t)**2)
        loss.backward(); _o.step()
        if ep%2000==0: free()
    set_grad(_p,False); set_grad(_s,True)
    _o2=optim.Adam(_s.parameters(), lr=1e-3)
    for ep in range(12000):
        _o2.zero_grad()
        x_c=x_col.detach().clone().requires_grad_(True)
        T_c=_p(x_c)
        dT=torch.autograd.grad(T_c,x_c,torch.ones_like(T_c),create_graph=True)[0]
        d2T=torch.autograd.grad(dT,x_c,torch.ones_like(dT),create_graph=False)[0]
        f_p=_s(x_col)
        x_r=x_col.detach().clone().requires_grad_(True)
        f_r=_s(x_r)
        df=torch.autograd.grad(f_r,x_r,torch.ones_like(f_r),create_graph=True)[0]
        loss=torch.mean((f_p+d2T.detach())**2) + wr*torch.mean(df**2)
        loss.backward(); _o2.step()
        if ep%2000==0: free()
    with torch.no_grad(): f_pred=_s(x_eval).squeeze().numpy().copy()
    l2=float(np.mean((f_pred-f_true(x_np))**2)**0.5)
    results_reg[wr]=(f_pred,l2)
    del _p,_s; free()
    print(f'  w_reg={wr:.0e}  →  f L² = {l2:.4f}')

plt.figure(figsize=(9, 5))
plt.plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$ true')
cmap2=plt.cm.viridis
for i,(wr,(fp,l2)) in enumerate(results_reg.items()):
    plt.plot(x_np, fp, '--', lw=1.8, color=cmap2(i/len(reg_weights)),
             label=f'$w_{{reg}}={wr:.0e}$ (L²={l2:.3f})')
plt.title('Effect of Tikhonov (noise=3%  —  correct range)', fontsize=12)
plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tikhonov_study.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free()

In [ ]:
# ══════════════════════════════════════════════
#  CELL 14 — Summary
# ══════════════════════════════════════════════
import glob
print('═'*55)
print('  RESULTS SUMMARY — v5')
print('═'*55)
print(f'  Temperature  L² error : {np.mean(T_err**2)**0.5:.2e}')
print(f'  Source       L² error : {np.mean(f_err**2)**0.5:.2e}')
print(f'  Observations           : {N_OBS} pts, noise={NOISE*100:.0f}%')
print()
print('  Key fixes vs v4:')
print('  • w_reg Phase 2 : 1e-3 → 1e-7  (prevents over-smoothing f)')
print('  • Phase 3       : no Tikhonov  (PDE coupling is sufficient)')
print('  • Diagnostic    : Tikhonov/Fit ratio plot added')
print('═'*55)
print('Generated files:')
for fp in sorted(glob.glob(f'{RESULTS_DIR}/*')):
    print(f'  {os.path.basename(fp):40s} {os.path.getsize(fp)//1024:>5} KB')
if IN_COLAB:
    print('\nSave to Drive:')
    print('  from google.colab import drive; drive.mount("/content/drive")')
    print('  !cp -r /content/results /content/drive/MyDrive/')